# Q3 University library help desk (SimPy)

Help desk with exponential service (mean 7 min) over a 360-minute session; target: 97% wait under 8 minutes.
Part (b): 2 advisers at 10 students/hour. Part (c): peak 15/hour with 2, 3, 4 advisers. Part (d): 10% priority users.

*Original script: `5 Source Files/Python/uni_library_simpy.py` (kept unchanged). Same code, split into steps, plus an
added section at the end for parts (c) and (e). Expected (b): 41 served, average wait 1.17 min, maximum 10.32 min,
95.12% under 8 minutes.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install simpy

In [2]:
import random

import simpy

**`basic_helpdesk_simulation`** (function): Standard queue for parts (b) and (c).

In [3]:
def basic_helpdesk_simulation():
    """Standard queue for parts (b) and (c)."""

    # Students can change this value and rerun the file.
    ADVISERS = 2

    ARRIVAL_RATE_PER_HOUR = 10
    MEAN_SERVICE_MINUTES = 7
    RUN_MINUTES = 360
    RANDOM_SEED = 24
    TARGET_WAIT = 8

    rng = random.Random(RANDOM_SEED)
    wait_times = []

    def arrivals(env, desk):
        while True:
            # Time until the next student arrives.
            interarrival = rng.expovariate(ARRIVAL_RATE_PER_HOUR / 60.0)
            yield env.timeout(interarrival)

            env.process(student(env, desk))

    def student(env, desk):
        arrival_time = env.now

        # The student joins the queue and waits for an adviser.
        with desk.request() as request:
            yield request

            wait = env.now - arrival_time
            wait_times.append(wait)

            # Once service starts, service time is random.
            service_time = rng.expovariate(1.0 / MEAN_SERVICE_MINUTES)
            yield env.timeout(service_time)

    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=ADVISERS)
    env.process(arrivals(env, desk))

    # Stop the simulation at the end of the session.
    env.run(until=RUN_MINUTES)

    served = len(wait_times)

    print("STANDARD MODEL")
    print("Advisers =", ADVISERS)
    print("Students served =", served)
    print("Average wait =", sum(wait_times) / served if served else 0.0)
    print("Maximum wait =", max(wait_times) if served else 0.0)
    print(
        "Percent waiting less than",
        TARGET_WAIT,
        "minutes =",
        100 * sum(wait < TARGET_WAIT for wait in wait_times) / served if served else 0.0,
    )
    print()

**`priority_helpdesk_simulation`** (function): Priority-user queue for part (d).

In [4]:
def priority_helpdesk_simulation():
    """Priority-user queue for part (d)."""

    # Students can change this value and rerun the file.
    ADVISERS = 3

    ARRIVAL_RATE_PER_HOUR = 15
    MEAN_SERVICE_MINUTES = 7
    RUN_MINUTES = 360
    RANDOM_SEED = 24
    PRIORITY_SHARE = 0.10
    TARGET_WAIT = 8

    rng = random.Random(RANDOM_SEED)
    all_waits = []
    priority_waits = []
    standard_waits = []

    def arrivals(env, desk):
        while True:
            interarrival = rng.expovariate(ARRIVAL_RATE_PER_HOUR / 60.0)
            yield env.timeout(interarrival)

            # Some students are marked as priority users.
            is_priority = rng.random() < PRIORITY_SHARE
            env.process(student(env, desk, is_priority))

    def student(env, desk, is_priority):
        arrival_time = env.now

        # Smaller number = higher priority in SimPy.
        if is_priority:
            priority_level = 0
        else:
            priority_level = 1

        with desk.request(priority=priority_level) as request:
            yield request

            wait = env.now - arrival_time
            all_waits.append(wait)

            if is_priority:
                priority_waits.append(wait)
            else:
                standard_waits.append(wait)

            service_time = rng.expovariate(1.0 / MEAN_SERVICE_MINUTES)
            yield env.timeout(service_time)

    env = simpy.Environment()
    desk = simpy.PriorityResource(env, capacity=ADVISERS)
    env.process(arrivals(env, desk))
    # Stop the simulation at the end of the session.
    env.run(until=RUN_MINUTES)

    served = len(all_waits)

    print("PRIORITY MODEL")
    print("Advisers =", ADVISERS)
    print("Students served =", served)
    print(
        "Overall percent waiting less than",
        TARGET_WAIT,
        "minutes =",
        100 * sum(wait < TARGET_WAIT for wait in all_waits) / served if served else 0.0,
    )
    print("Priority average wait =", sum(priority_waits) / len(priority_waits) if priority_waits else 0.0)
    print("Standard average wait =", sum(standard_waits) / len(standard_waits) if standard_waits else 0.0)
    print()

**Run it.** In a notebook `__name__` is `"__main__"`, so this block runs everything above.

In [5]:
if __name__ == "__main__":
    basic_helpdesk_simulation()
    priority_helpdesk_simulation()

STANDARD MODEL
Advisers = 2
Students served = 41
Average wait = 1.1725005806444826
Maximum wait = 10.315086829142388
Percent waiting less than 8 minutes = 95.1219512195122

PRIORITY MODEL
Advisers = 3
Students served = 86
Overall percent waiting less than 8 minutes = 96.51162790697674
Priority average wait = 0.4480592978962932
Standard average wait = 0.608401197961288



## Part (c) as a table, and part (e): replications (added)

`basic_helpdesk_simulation` has its settings written inside the function, so part (c) means editing `ADVISERS` and the
arrival rate and re-running. The function below is the **same model** with the settings as arguments. The first table
reproduces the Solutions PDF table (seed 24). The second table repeats each case with 200 seeds, which is what part (e) asks for.

In [6]:
def helpdesk(advisers, arrivals_per_hour, seed=24, minutes=360, mean_service=7, target=8):
    """Same model as basic_helpdesk_simulation, with the settings as arguments."""
    rng = random.Random(seed)
    waits = []

    def arrivals(env, desk):
        while True:
            yield env.timeout(rng.expovariate(arrivals_per_hour / 60.0))
            env.process(student(env, desk))

    def student(env, desk):
        arrival_time = env.now
        with desk.request() as request:
            yield request
            waits.append(env.now - arrival_time)
            yield env.timeout(rng.expovariate(1.0 / mean_service))

    env = simpy.Environment()
    desk = simpy.Resource(env, capacity=advisers)
    env.process(arrivals(env, desk))
    env.run(until=minutes)
    w = waits
    return len(w), sum(w) / len(w), max(w), 100 * sum(x < target for x in w) / len(w)

print("Peak period (15 students/hour), seed 24")
print(f"{'Advisers':<10}{'Served':>8}{'Avg wait':>10}{'Max wait':>10}{'% < 8 min':>11}")
for a in (2, 3, 4):
    n, avg, mx, pct = helpdesk(a, 15)
    print(f"{a:<10}{n:>8}{avg:>10.2f}{mx:>10.2f}{pct:>10.2f}%")

Peak period (15 students/hour), seed 24
Advisers    Served  Avg wait  Max wait  % < 8 min
2               70      1.88     12.99     95.71%
3               73      0.32      3.22    100.00%
4               70      0.08      2.23    100.00%


In [7]:
import statistics
reps = 200
print(f"Peak period, {reps} replications (seeds 0-{reps - 1})")
print(f"{'Advisers':<10}{'Mean % < 8 min':>16}{'Runs meeting 97%':>18}")
for a in (2, 3, 4):
    pcts = [helpdesk(a, 15, seed=s)[3] for s in range(reps)]
    print(f"{a:<10}{statistics.mean(pcts):>15.1f}%{100 * sum(p >= 97 for p in pcts) / reps:>17.0f}%")

Peak period, 200 replications (seeds 0-199)
Advisers    Mean % < 8 min  Runs meeting 97%


2                    54.9%                1%


3                    94.0%               52%


4                    99.3%               92%


**Reading it.** With seed 24, 3 advisers meet the 97% target, as the Solutions PDF says. Seed 24 is a light day,
though: only about 70 students are served, against 90 expected arrivals (15/hour × 6 h). Over 200 days, 3 advisers average about 94% under
8 minutes and meet the target on only about half the days. 4 advisers average about 99%. The Erlang-C formula agrees:
92.0% for 3 advisers and 99.1% for 4. This is open finding C17 in the Week 10 README; the Solutions PDF is unchanged.
"Served" counts students who *started* service before closing time. Anyone still queueing at 360 minutes is not counted.